In [1]:
# This notebook integrates DHS datasets to extract required features for modelling
import duckdb
import pandas as pd
import numpy as np
import geopandas as gpd

# Load DTA file
dhs_INHR = "DHSData/India/IAHR7EDT/IAHR7EFL.DTA"
dhs_INMR = "DHSData/India/IAMR7EDT/IAMR7EFL.DTA"
dhs_INIR = "DHSData/India/IAIR7EDT/IAIR7EFL.DTA"

iahr_use_cols=[
    'hhid','hv000','hv001','hv002', #hv000-country code and phase, hv001-cluster number, hv002-household number
    'hv005', # simpliling weight
    'hv270', # wealth index
    'hv012', #Whether the member is a de jure household member, i.e., whether the member is a usual resident of the household.
    'hv013', #Whether the member is a de facto household member, i.e., whether the member slept in the household the previous night.
     # MEPI indicators 
    'hv206',     # electricity
    'hv226',     # cooking fuel
    'hv241',     # kitchen type food cooked in the house
    'hv242',     # household has separate room used as kitchen yes no
    'hv209',     # refrigerator
    'hv221',     # landline
    'hv243a',    # mobile phone
    'hv208',     # television
    'hv207',     # radio
    'hv213',    #floor material
    'hv215',    #root material

    # socio-demographic variables
    'hv024',    # region division
    'hv025',    # type of place of residence urban/rural
    'hv219',    # sex of head
    'hv220',    # age of head
    'hv106_01', # education of head
    'hv115_01', # marital status
    'hv009',    # family side members of the household
    'hv247',   # bank account
    'hv216',   # household size rooms used for sleeping
    'hv014'   # children under 5 (de jure)
]

iamr_use_cols = [
'mcaseid', #case identification
'mv000',	#country code and phase
'mv001',	#cluster number
'mv002',	#household number
'mv150',	#relationship to household head
'mv714',	#currently working
'mv717',	#occupation (grouped)
'mv745a', #	owns a house alone or jointly
]

iair_use_cols = [
'caseid', #case identification
'v000',	#country code and phase
'v001',	#cluster number
'v002',	#household number
'v150',	#relationship to household head
'v714',	#currently working
'v717',	#occupation (grouped)
'v745a'  #owns a house alone or jointly
]

df_INMR = pd.read_stata(dhs_INMR, columns=iamr_use_cols, convert_categoricals=False)

df_INIR = pd.read_stata(dhs_INIR, columns=iair_use_cols,convert_categoricals=False)

df_INHR = pd.read_stata(dhs_INHR, columns=iahr_use_cols, convert_categoricals=False)

# Convert problematic df_HR numpy arrays to regular Python types
for col in df_INHR.columns:
    if df_INHR[col].dtype == object:
        # Convert object columns that might contain numpy arrays
        df_INHR[col] = df_INHR[col].astype(str)
    elif isinstance(df_INHR[col].dtype, pd.CategoricalDtype):
        # Convert categorical to string
        df_INHR[col] = df_INHR[col].astype(str)
        
# Convert problematic df_MR numpy arrays to regular Python types
for col in df_INMR.columns:
    if df_INMR[col].dtype == object:
        # Convert object columns that might contain numpy arrays
        df_INMR[col] = df_INMR[col].astype(str)
    elif isinstance(df_INMR[col].dtype, pd.CategoricalDtype):
        # Convert categorical to string
        df_INMR[col] = df_INMR[col].astype(str)

# Convert problematic df_IR numpy arrays to regular Python types
for col in df_INHR.columns:
    if df_INHR[col].dtype == object:
        # Convert object columns that might contain numpy arrays
        df_INHR[col] = df_INHR[col].astype(str)
    elif isinstance(df_INHR[col].dtype, pd.CategoricalDtype):
        # Convert categorical to string
        df_INHR[col] = df_INHR[col].astype(str)


df_IN_HR_MR_IR = duckdb.sql("""
SELECT 
hhid AS 'hhidCaseIdentification',
hv000 AS 'hv000CountryCode',
hv001 AS 'hv001ClusterNumber',
hv002 AS 'hv002HouseholdNumber',
hv005 AS 'hv005SimplilingWeight',
hv270 AS 'hv270WealthIndex',
hv206 AS 'hv206HasElectricity',
hv226 AS 'hv226CookingFuel',
hv241 AS 'hv241FoodCookedInHouse',
hv242 AS 'hv242SeparateKitchenYesNo',
hv209 AS 'hv209HasRefrigerator',
hv221 AS 'hv221HasLandLine',
hv243a AS 'hv243aHasMobilePhone',
hv208 AS 'hv208HasTelevision',
hv024 AS 'hv024RegionDivision', 
hv025 AS 'hv025TypeOfResidence', 
hv219 AS 'hv219SexOfHead',
hv220 AS 'hv220AgeOfHead',
hv106_01 AS 'hv106_01EducationOfHead',
hv115_01 AS 'hv115_01MaritalStatus',
hv009 AS 'hv009FamilySize',
hv247 AS 'hv247HasBankAccount',
hv216 AS 'hv216HouseSize',  
hv014 AS 'hv014NoOfChildren'
,CASE 
        WHEN HR.hv220 BETWEEN 15 AND 49 AND HR.hv219 = 1 THEN COALESCE(MR.mv714, 0)
        WHEN HR.hv220 BETWEEN 15 AND 49 AND HR.hv219 = 2 THEN COALESCE(IR.v714, 0)
        ELSE 0
    END AS 'v714CurrentlyWorking',
    CASE 
        WHEN HR.hv220 BETWEEN 15 AND 49 AND HR.hv219 = 1 THEN COALESCE(MR.mv717, 0)
        WHEN HR.hv220 BETWEEN 15 AND 49 AND HR.hv219 = 2 THEN COALESCE(IR.v717, 0)
        ELSE 0
    END AS 'v717Occupation',
    CASE 
        WHEN HR.hv220 BETWEEN 15 AND 49 AND HR.hv219 = 1 THEN COALESCE(MR.mv745a, 0)
        WHEN HR.hv220 BETWEEN 15 AND 49 AND HR.hv219 = 2 THEN COALESCE(IR.v745a, 0)
        ELSE 0
    END AS '745aHouseOwnership'
FROM df_INHR HR 
LEFT JOIN df_INMR MR 
    ON HR.hv001 = MR.mv001 
    AND HR.hv002 = MR.mv002
    AND MR.mv150 = 1  -- Man is household head
    AND HR.hv219 = 1  -- Head is male
LEFT JOIN df_INIR IR 
    ON HR.hv001 = IR.v001 
    AND HR.hv002 = IR.v002
    AND IR.v150 = 1   -- Woman is household head
    AND HR.hv219 = 2  -- Head is female
""").df()

df_IN_HR_MR_IR.head()


,hhidCaseIdentification,hv000CountryCode,hv001ClusterNumber,hv002HouseholdNumber,hv005SimplilingWeight,hv270WealthIndex,hv206HasElectricity,hv226CookingFuel,hv241FoodCookedInHouse,hv242SeparateKitchenYesNo,...,hv220AgeOfHead,hv106_01EducationOfHead,hv115_01MaritalStatus,hv009FamilySize,hv247HasBankAccount,hv216HouseSize,hv014NoOfChildren,v714CurrentlyWorking,v717Occupation,745aHouseOwnership
0,0100103079,IA7,130,79,204093,3,1,2,1.0,1.0,...,42,2,1.0,4,1,2,0,1.0,6.0,3.0
1,0100103033,IA7,130,33,204093,3,1,2,1.0,1.0,...,42,1,1.0,6,1,2,1,1.0,4.0,3.0
2,0100103089,IA7,130,89,204093,3,1,2,1.0,1.0,...,43,2,1.0,5,1,2,0,1.0,6.0,3.0
3,0100103075,IA7,130,75,204093,3,1,2,1.0,1.0,...,54,3,1.0,4,1,3,0,0.0,0.0,0.0
4,0100100997,IA7,109,97,199431,3,1,2,1.0,1.0,...,54,2,1.0,3,1,2,0,0.0,0.0,0.0


In [2]:
df_IN_HR_MR_IR["weight"] = df_IN_HR_MR_IR["hv005SimplilingWeight"] / 1_000_000

In [3]:
import geopandas as gpd
dhs_INGE = "DHSData/INDIA/IAGE7AFL.dbf"

iage_use_cols =['ALT_DEM', #elevation from DEM (meters)
      'DHSCLUST' #cluster ID (this matches hv001 / v001)
      ]

df_INGE = gpd.read_file(dhs_INGE,columns=iage_use_cols)

#join with  Cluster level Elevation
df_INresult = duckdb.sql("""
SELECT
    H.*,
    NULLIF(G.ALT_DEM, 9999) AS HouseholdClusterElevation
FROM df_IN_HR_MR_IR AS H
LEFT JOIN df_INGE AS G
    ON H.hv001ClusterNumber = G.DHSCLUST
""").df()

df_INresult.head()

#save data into parquet
df_INresult.to_parquet(
    "INDIA/DATA/1.df_IN_NecessaryColumns.parquet",
    engine="pyarrow",
    compression="snappy"
)
print(f"✅ File saved.")


✅ File saved.
